In [4]:
"""
Cross-Reference MLflow and W&B Results
"""

import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlflow.tracking import MlflowClient
import wandb

# ------------------------- MLflow Analysis -------------------------
print("="*50)
print("MLFLOW ANALYSIS")
print("="*50)

# Set tracking URI
mlflow.set_tracking_uri('sqlite:///mlflow.db')

# Get all runs
experiment = mlflow.get_experiment_by_name("weed-detection")
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Display runs summary
print("\n📊 All Runs Summary:")
display_cols = ['run_id', 'metrics.val_acc', 'metrics.val_loss', 
                'metrics.train_acc', 'metrics.train_loss',
                'params.learning_rate', 'params.architecture',
                'params.use_focal_loss', 'tags.mlflow.runName']
available_cols = [col for col in display_cols if col in runs_df.columns]
print(runs_df[available_cols].to_string())

# Find best run by validation accuracy
best_run = runs_df.loc[runs_df['metrics.val_acc'].idxmax()]
print(f"\n🏆 Best Run (by val_acc):")
print(f"Run ID: {best_run['run_id']}")
print(f"Run Name: {best_run.get('tags.mlflow.runName', 'N/A')}")
print(f"Val Acc: {best_run['metrics.val_acc']:.4f}")
print(f"Val Loss: {best_run['metrics.val_loss']:.4f}")
print(f"Train Acc: {best_run['metrics.train_acc']:.4f}")
print(f"Learning Rate: {best_run['params.learning_rate']}")

# Plot comparison of runs
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Validation Accuracy comparison
ax1 = axes[0, 0]
runs_df_sorted = runs_df.sort_values('metrics.val_acc', ascending=False)
ax1.barh(range(len(runs_df_sorted)), runs_df_sorted['metrics.val_acc'].values)
ax1.set_yticks(range(len(runs_df_sorted)))
ax1.set_yticklabels(runs_df_sorted['tags.mlflow.runName'].values if 'tags.mlflow.runName' in runs_df_sorted.columns else runs_df_sorted['run_id'].str[:8])
ax1.set_xlabel('Validation Accuracy')
ax1.set_title('MLflow: Validation Accuracy by Run')

# Validation Loss comparison
ax2 = axes[0, 1]
ax2.barh(range(len(runs_df_sorted)), runs_df_sorted['metrics.val_loss'].values)
ax2.set_yticks(range(len(runs_df_sorted)))
ax2.set_yticklabels(runs_df_sorted['tags.mlflow.runName'].values if 'tags.mlflow.runName' in runs_df_sorted.columns else runs_df_sorted['run_id'].str[:8])
ax2.set_xlabel('Validation Loss')
ax2.set_title('MLflow: Validation Loss by Run')

# Scatter plot: val_acc vs val_loss
ax3 = axes[1, 0]
scatter = ax3.scatter(runs_df['metrics.val_loss'], runs_df['metrics.val_acc'],
                      c=range(len(runs_df)), cmap='viridis', s=100)
ax3.set_xlabel('Validation Loss')
ax3.set_ylabel('Validation Accuracy')
ax3.set_title('MLflow: Val Acc vs Val Loss')
plt.colorbar(scatter, ax=ax3, label='Run Index')

# Learning Rate vs Val Acc
ax4 = axes[1, 1]
if 'params.learning_rate' in runs_df.columns:
    runs_df['params.learning_rate'] = pd.to_numeric(runs_df['params.learning_rate'], errors='coerce')
    ax4.scatter(runs_df['params.learning_rate'], runs_df['metrics.val_acc'], s=100)
    ax4.set_xlabel('Learning Rate')
    ax4.set_ylabel('Validation Accuracy')
    ax4.set_title('MLflow: Learning Rate vs Val Acc')
    ax4.set_xscale('log')

plt.tight_layout()
plt.savefig('mlflow_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------- Per-Class Performance Analysis -------------------------
print("\n📈 Per-Class Performance Analysis:")
class_cols = [col for col in runs_df.columns if col.startswith('metrics.cls_acc_')]
if class_cols:
    class_data = runs_df[['tags.mlflow.runName'] + class_cols].copy()
    class_data.columns = [col.replace('metrics.cls_acc_', '') if col.startswith('metrics.cls_acc_') else col 
                          for col in class_data.columns]
    
    # Melt for easier plotting
    class_melted = class_data.melt(id_vars=['tags.mlflow.runName'], 
                                    var_name='Class', value_name='Accuracy')
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=class_melted, x='Class', y='Accuracy', hue='tags.mlflow.runName')
    plt.title('Per-Class Accuracy Across Runs')
    plt.xticks(rotation=45)
    plt.legend(title='Run Name', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()

# ------------------------- W&B Integration -------------------------
print("\n" + "="*50)
print("W&B CROSS-REFERENCE")
print("="*50)

# You can also query W&B API if you have wandb installed and logged in
try:
    api = wandb.Api()
    
    # Get your project runs
    runs = api.runs("kumardahal788-loyalist-college-of-toronto/weed-detection")
    
    print("\n📊 W&B Runs Summary:")
    for run in runs:
        print(f"\nRun: {run.name}")
        print(f"  State: {run.state}")
        print(f"  Val Acc: {run.summary.get('val/acc', 'N/A')}")
        print(f"  Train Acc: {run.summary.get('train/acc', 'N/A')}")
        print(f"  URL: {run.url}")
        
        # Find matching MLflow run
        mlflow_match = runs_df[runs_df['tags.wandb_run_id'] == run.id]
        if not mlflow_match.empty:
            print(f"  ✅ MLflow Run ID: {mlflow_match.iloc[0]['run_id']}")
        else:
            print(f"  ❌ No matching MLflow run found")
            
except Exception as e:
    print(f"W&B API error: {e}")
    print("Make sure you're logged in to wandb: wandb login")

# ------------------------- Generate Comparison Report -------------------------
print("\n" + "="*50)
print("COMPARISON REPORT")
print("="*50)

report = {
    'total_runs': len(runs_df),
    'best_val_acc': runs_df['metrics.val_acc'].max(),
    'best_val_acc_run': runs_df.loc[runs_df['metrics.val_acc'].idxmax(), 'tags.mlflow.runName'] 
                        if 'tags.mlflow.runName' in runs_df.columns else 'N/A',
    'avg_val_acc': runs_df['metrics.val_acc'].mean(),
    'std_val_acc': runs_df['metrics.val_acc'].std(),
    'best_val_loss': runs_df['metrics.val_loss'].min(),
    'avg_val_loss': runs_df['metrics.val_loss'].mean(),
}

print("\n📋 Summary Statistics:")
for key, value in report.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Save report
import json
with open('training_comparison_report.json', 'w') as f:
    json.dump({k: float(v) if isinstance(v, (float, int)) else v for k, v in report.items()}, f, indent=2)

print("\n✅ Report saved to training_comparison_report.json")
print("📊 MLflow plots saved to mlflow_analysis.png")
print("📈 Per-class accuracy plot saved to per_class_accuracy.png")

MLFLOW ANALYSIS


AttributeError: 'NoneType' object has no attribute 'experiment_id'

In [5]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

experiments = mlflow.search_experiments()

for exp in experiments:
    print(exp.name, exp.experiment_id)

Default 0
